In [ ]:
import pandas as pd
import numpy as np
from utils import reduce_memory_usage

# 1. 데이터 로드
print("기초 데이터 로드 중...")
orders = pd.read_csv('../data/raw/orders.csv')
prior_details = pd.read_csv('../data/raw/order_products__prior.csv')
products = pd.read_csv('../data/raw/products.csv') # aisle_id, department_id 추출용

# 2. 유저-상품 기본 조합 생성 (user_id 주입)
# prior_details에는 user_id가 없으므로 orders와 결합하여 가져옵니다.
print("유저-상품 기본 데이터 결합 중...")
up_base = prior_details.merge(orders[['order_id', 'user_id']], on='order_id', how='left')

# 3. 피처 계산
print("유저-상품별 파생변수 계산 중...")
user_prod_features = up_base.groupby(['user_id', 'product_id']).agg(
    prod_reorder_cnt=('reordered', 'sum'),
    total_buy_cnt=('reordered', 'count'), # 비율 계산용 임시 컬럼
    first_cart_cnt=('add_to_cart_order', lambda x: (x == 1).sum()),
    cart_top5_cnt=('add_to_cart_order', lambda x: (x <= 5).sum()),
    prod_cart_std=('add_to_cart_order', 'std')
).reset_index()

# 4. 비율 및 추가 피처 계산
# (1) 해당 상품의 비-재구매(첫 구매) 횟수
user_prod_features['prod_non_reorder_cnt'] = user_prod_features['total_buy_cnt'] - user_prod_features['prod_reorder_cnt']
# (2) 첫 번째로 장바구니에 담긴 비율
user_prod_features['prod_first_cart_rate'] = user_prod_features['first_cart_cnt'] / user_prod_features['total_buy_cnt']
# (3) 5번째 이내로 장바구니에 담긴 비율
user_prod_features['prod_cart_top5_rate'] = user_prod_features['cart_top5_cnt'] / user_prod_features['total_buy_cnt']
# (4) 재구매 평형(재구매 비율)
user_prod_features['prod_reorder_balance'] = user_prod_features['prod_reorder_cnt'] / user_prod_features['total_buy_cnt']

# 5. 카테고리 정보(aisle, department) 결합
user_prod_features = user_prod_features.merge(products[['product_id', 'aisle_id', 'department_id']], on='product_id', how='left')

# 6. 중간 계산용 임시 컬럼 제거 및 메모리 최적화
user_prod_features.drop(columns=['total_buy_cnt', 'first_cart_cnt', 'cart_top5_cnt'], inplace=True)
user_prod_features = reduce_memory_usage(user_prod_features)

# 7. 결과 저장
output_path = '../data/prep/user-prod_features.csv'
user_prod_features.to_csv(output_path, index=False)

print(f"생성 완료: {output_path}")
print("포함된 컬럼:", user_prod_features.columns.tolist())

In [ ]:
import pandas as pd
from utils import reduce_memory_usage

# 파일 로드
order_feat = pd.read_csv('../data/prep/order_features.csv')
prod_feat = pd.read_csv('../data/prep/prod_features.csv')
user_feat = pd.read_csv('../data/prep/user_features.csv')
user_prod_feat = pd.read_csv('../data/prep/user-prod_features.csv')

# 원본 데이터 로드 (유저가 과거에 샀던 상품 리스트를 만들기 위해 필요)
orders = pd.read_csv('../data/raw/orders.csv')
prior_details = pd.read_csv('../data/raw/order_products__prior.csv')
train_actual = pd.read_csv('../data/raw/order_products__train.csv')

# 유저-상품 후보군(Candidate) 생성
# '어떤 유저가 과거(prior)에 어떤 상품을 샀었는가'가 모델의 학습 데이터(Row)
user_product_candidates = prior_details.merge(orders[['order_id', 'user_id']], on='order_id')
user_product_candidates = user_product_candidates[['user_id', 'product_id']].drop_duplicates()

# 기준 데이터셋 생성 (eval_set이 train/test인 주문)
data = orders[orders.eval_set != 'prior'][['user_id', 'order_id', 'eval_set', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']]
data.rename(columns={
    'order_hour_of_day': 'order_hour',
    'days_since_prior_order': 'days_since_prior'
    }, inplace=True)

# 후보군과 주문 정보 결합 
data = data.merge(user_product_candidates, on='user_id', how='left')

# 파생변수 결합 (Merge)
def merge_without_duplicates(base_df, feature_df, on_cols):
    """중복 컬럼 발생을 방지하며 Merge하는 함수"""
    # 결합 기준 컬럼(on_cols)을 제외하고, base_df에 이미 존재하는 컬럼은 feature_df에서 제거
    cols_to_use = feature_df.columns.difference(base_df.columns).tolist() + on_cols
    return base_df.merge(feature_df[cols_to_use], on=on_cols, how='left')

# (1) 유저 특성 (user_id 기준)
data = merge_without_duplicates(data, user_feat, ['user_id'])

# (2) 상품 특성 (product_id 기준)
data = merge_without_duplicates(data, prod_feat, ['product_id'])

# (3) 유저-상품 결합 특성 (user_id, product_id 기준)
data = merge_without_duplicates(data, user_prod_feat, ['user_id', 'product_id'])

# (4) 주문 특성 (order_id 기준)
data = merge_without_duplicates(data, order_feat, ['order_id'])

# 정답지(Label) 붙이기
train_actual = pd.read_csv('../data/raw/order_products__train.csv')
data = data.merge(train_actual[['order_id', 'product_id', 'reordered']], 
                  on=['order_id', 'product_id'], how='left')

# 실제 주문하지 않은 상품은 재구매(1)가 아니므로 0으로 채움
data['reordered'] = data['reordered'].fillna(0)

# 중복 컬럼 제거
cols_to_drop = [c for c in data.columns if c.endswith(('_extra', '_order', '_x', '_y'))]
if cols_to_drop:
    print(f"제거된 중복 컬럼: {cols_to_drop}")
    data.drop(columns=cols_to_drop, inplace=True)

# 메모리 최적화
data = reduce_memory_usage(data)
print(f"최종 컬럼 수: {len(data.columns)}")
print(f"최종 데이터 형태: {data.shape}")

# 통합 데이터 csv 파일로 저장
output_path = '../data/prep/total_data.csv'
data.to_csv(output_path, index=False)